# Stage 6-4: Regression 실험 비교

이 노트북은 `experiments/run_all.py`로 실행한 regression task의 MLP와 CNN 학습 결과를 비교한다.

**Regression task 특징**: MNIST 레이블을 `label / 9.0`으로 정규화하여 [0, 1] 연속값 예측을 수행한다.
예측값은 `round(pred * 9)`로 복원하여 원래 숫자(0~9)와 비교한다.

**실험 설정**
| 모델 | exp_name |
|---|---|
| MLP | `regression_mlp_ep10_lr0.01_bs64` |
| CNN | `regression_cnn_ep10_lr0.001_bs32` |

In [ ]:
# sys.path 설정 — 프로젝트 루트를 모듈 검색 경로에 추가한다.
import sys, os
sys.path.insert(0, os.path.abspath("../.."))

In [ ]:
import subprocess
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import tempfile

In [ ]:
from src.data.mnist import MNISTDataset, get_task_spec
from src.data.dataloader import Dataloader
from src.models.mlp import MLP
from src.core.evaluator import Evaluator
from src.core.predictor import Predictor
from src.core.visualizer import Visualizer
from src.utils import checkpoints

In [ ]:
# 학습 결과 없으면 run_all.py 실행
TASK = "regression"
MLP_DIR = "../../outputs/regression_mlp_ep10_lr0.01_bs64"
CNN_DIR = "../../outputs/regression_cnn_ep10_lr0.001_bs32"
DATASET_DIR = "/mnt/d/datasets/mnist"

if not os.path.exists(f"{MLP_DIR}/model.npz"):
    print("학습 결과 없음 — run_all.py 실행 중... (수 분 소요)")
    r = subprocess.run(
        [sys.executable, "../../experiments/run_all.py"],
        capture_output=True, text=True,
    )
    print(r.stdout[-1000:])
else:
    print("학습 결과 확인:")
    print(f"  MLP: {MLP_DIR}/model.npz")
    cnn_status = "있음" if os.path.exists(f"{CNN_DIR}/model.npz") else "없음 (CuPy 환경 필요)"
    print(f"  CNN: {cnn_status}")

## 1. Regression 개요

MNIST 레이블을 `label / 9.0`으로 정규화하여 연속값을 예측하는 task이다.
예측값에 9를 곱하고 반올림하여 0~9 정수로 복원한다.

| 항목 | 내용 |
|---|---|
| 출력 차원 | 1 (연속값) |
| 타겟 변환 | `label / 9.0` → [0, 1] |
| loss | MSE |
| metric | R² score |
| 후처리 | `clip(round(pred * 9), 0, 9)` |

**round_clip 후처리**:
$$\text{prediction} = \text{clip}(\text{round}(\hat{y} \times 9),\ 0,\ 9)$$

정규화된 예측값 $\hat{y}$에 9를 곱해 원래 스케일로 복원하고, 반올림 후 [0, 9]로 clip한다.

In [ ]:
# task_spec 및 데이터 확인
ts = get_task_spec(TASK)
print("task_spec:", ts)

test_ds = MNISTDataset("test", TASK, dataset_dir=DATASET_DIR)
print(f"test set: {len(test_ds)} samples")
print(f"타겟 범위: [{test_ds.targets.min():.3f}, {test_ds.targets.max():.3f}] (label/9.0 정규화)")
print(f"labels_raw 범위: [{test_ds.labels_raw.min()}, {test_ds.labels_raw.max()}] (원래 레이블)")

In [ ]:
# round_clip 후처리 검증
pred_values = np.array([-0.1, 0.0, 0.5, 1.0, 1.1], dtype=np.float32)
decoded = np.clip(np.round(pred_values * 9), 0, 9).astype(int)
print("예측값:", pred_values)
print("round_clip 결과:", decoded)

## 2. MLP 결과 — 학습 곡선과 예측 grid

In [ ]:
mlp_log = f"{MLP_DIR}/training_log.png"
mlp_pred = f"{MLP_DIR}/predictions.png"

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, img_path, title in [
    (axes[0], mlp_log, "MLP: Training Log"),
    (axes[1], mlp_pred, "MLP: Predictions Grid"),
]:
    if os.path.exists(img_path):
        ax.imshow(Image.open(img_path))
    else:
        ax.text(0.5, 0.5, "결과 없음", ha="center", va="center")
    ax.set_title(title)
    ax.axis("off")

plt.suptitle("Regression MLP (10 epochs, lr=0.01, bs=64)", fontsize=13)
plt.tight_layout()
plt.show()

## 3. MLP checkpoint 로드 후 직접 평가

In [ ]:
test_loader = Dataloader(test_ds, batch_size=256, shuffle=False)

mlp_model = MLP(task=TASK, seed=42)
if os.path.exists(f"{MLP_DIR}/model.npz"):
    checkpoints.load(mlp_model, f"{MLP_DIR}/model.npz")
    print("MLP checkpoint 로드 완료")

evaluator_mlp = Evaluator(mlp_model, ts)
result_mlp = evaluator_mlp.evaluate(test_loader)
print(f"MLP test: MSE loss={result_mlp['loss']:.6f}, R2={result_mlp['metric']:.4f}, samples={result_mlp['num_samples']}")

In [ ]:
# 예측 grid 직접 생성
tmpdir = tempfile.mkdtemp()
images_sample = test_ds.images[:32]
labels_sample = test_ds.labels_raw[:32]

predictor_mlp = Predictor(mlp_model, ts)
pred_mlp = predictor_mlp.predict(images_sample)

vis = Visualizer(output_dir=tmpdir)
grid_path = vis.plot_predictions(
    images=images_sample,
    labels=labels_sample,
    predictions=pred_mlp["predictions"],
    filename="regression_mlp_preds.png",
    n=16,
)

print("예측값 (round_clip):", pred_mlp["predictions"][:8])
print("실제 레이블:         ", labels_sample[:8])

plt.figure(figsize=(14, 4))
plt.imshow(Image.open(grid_path))
plt.axis("off")
plt.title("Regression MLP 예측 grid (round_clip 후처리)")
plt.tight_layout()
plt.show()

## 4. R² score 해석

$$R^2 = 1 - \frac{\sum_i (y_i - \hat{y}_i)^2}{\sum_i (y_i - \bar{y})^2}$$

- $R^2 = 1.0$: 완벽한 예측
- $R^2 = 0.0$: 평균값만 예측하는 것과 같음
- $R^2 < 0.0$: 평균값보다 나쁜 예측

MSE loss는 정규화된 [0, 1] 스케일에서 계산되므로 작은 값으로 나타난다.

In [ ]:
# R2 score 계산 원리 확인
from src.nn.metrics import r2_score

# 완벽한 예측
y_true = np.array([0.0, 1/9, 2/9, 3/9, 4/9], dtype=np.float32).reshape(-1, 1)
y_pred_perfect = y_true.copy()
y_pred_mean = np.full_like(y_true, y_true.mean())
y_pred_rand = np.random.default_rng(0).random((5, 1)).astype(np.float32)

print("완벽한 예측 R2:", r2_score(y_pred_perfect, y_true))
print("평균 예측 R2:  ", r2_score(y_pred_mean, y_true))
print("랜덤 예측 R2:  ", r2_score(y_pred_rand, y_true))

## 5. MLP vs CNN 비교

In [ ]:
cnn_log = f"{CNN_DIR}/training_log.png"
cnn_pred = f"{CNN_DIR}/predictions.png"

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for i, (img_path, title) in enumerate([
    (mlp_log,  "MLP: Training Log (lr=0.01, bs=64)"),
    (cnn_log,  "CNN: Training Log (lr=0.001, bs=32)"),
    (mlp_pred, "MLP: Predictions Grid"),
    (cnn_pred, "CNN: Predictions Grid"),
]):
    ax = axes[i // 2][i % 2]
    if os.path.exists(img_path):
        ax.imshow(Image.open(img_path))
    else:
        ax.text(0.5, 0.5, "결과 없음", ha="center", va="center", fontsize=14)
    ax.set_title(title)
    ax.axis("off")

plt.suptitle("Regression: MLP vs CNN 비교", fontsize=14)
plt.tight_layout()
plt.show()

## 6. 3 task 최종 비교 정리

In [ ]:
# 3 task MLP 평가 결과 비교 (checkpoint가 있는 경우)
task_results = {}
task_configs = [
    ("multiclass", "multiclass_mlp_ep10_lr0.01_bs64"),
    ("binary",     "binary_mlp_ep10_lr0.01_bs64"),
    ("regression", "regression_mlp_ep10_lr0.01_bs64"),
]

for task, exp in task_configs:
    ckpt = f"../../outputs/{exp}/model.npz"
    if not os.path.exists(ckpt):
        print(f"  [{task}] checkpoint 없음")
        continue
    ts_t = get_task_spec(task)
    ds_t = MNISTDataset("test", task, dataset_dir=DATASET_DIR)
    loader_t = Dataloader(ds_t, batch_size=256, shuffle=False)
    m_t = MLP(task=task, seed=42)
    checkpoints.load(m_t, ckpt)
    ev_t = Evaluator(m_t, ts_t)
    res = ev_t.evaluate(loader_t)
    task_results[task] = res
    print(f"  [{task}] loss={res['loss']:.4f}, metric={res['metric']:.4f}")

In [ ]:
# 3 task metric 막대 그래프 (결과가 있는 경우)
if task_results:
    tasks = list(task_results.keys())
    metrics = [task_results[t]["metric"] for t in tasks]
    metric_labels = {"multiclass": "accuracy", "binary": "binary_accuracy", "regression": "R2 score"}

    fig, ax = plt.subplots(figsize=(8, 5))
    bars = ax.bar(tasks, metrics, color=["steelblue", "orange", "green"])
    ax.bar_label(bars, fmt="{:.4f}")
    ax.set_ylim(0, 1.1)
    ax.set_ylabel("Metric")
    ax.set_title("3 Task MLP (10 epochs) Test Metric 비교")
    for i, task in enumerate(tasks):
        ax.text(i, -0.08, metric_labels[task], ha="center", fontsize=9, color="gray")
    plt.tight_layout()
    plt.show()
else:
    print("결과 없음 — run_all.py 실행 필요")

## 7. 요약

**Regression task**
- 타겟 변환: `label / 9.0` → [0, 1] 연속값
- 후처리: `clip(round(pred * 9), 0, 9)` — 정수 복원
- metric: R² score — 1에 가까울수록 좋은 예측
- MSE loss는 정규화 스케일에서 계산되어 작은 값으로 나타남

**3 Task 비교**

| Task | loss 함수 | metric | 후처리 |
|---|---|---|---|
| multiclass | cross-entropy (softmax 내장) | accuracy | argmax |
| binary | BCE (sigmoid 내장) | binary_accuracy | threshold |
| regression | MSE | R² score | round_clip |

세 task 모두 동일한 `Trainer`, `Evaluator`, `Predictor` 인터페이스로 처리되며,
`task_spec`만 바꾸면 된다. 이것이 이 프로젝트의 핵심 설계 원칙이다.